# Main.py for testing


In [ ]:
from models.GAT.gat_journey_planner import create_pyg_graph
from pandasql import sqldf
import pandas as pd

def  load_pyg_data_t(start_des, final_des):
    pro_dir = "C:\\Users\\sasab\\Documents\\Projects\\MaaS_AI\\Main_App\\"
    data_path = "models\\GAT\\data\\"
    metro_edges_path = pro_dir+data_path+"pune_maas_journey_planner_data.csv"
    station_features_path = pro_dir+data_path+"station_features.csv"
    
    
    #loading in csv file
    route_edges_df = pd.read_csv(metro_edges_path) #pune_maas_journey_planner_data.csv - metro routing data table
    station_feat_df = pd.read_csv(station_features_path) #timetable_data.csv - metro train timetable data

    # clean columns data
    route_edges_df.columns = route_edges_df.columns.str.strip()
    station_feat_df.columns = station_feat_df.columns.str.strip()

    # fix differnt name - normalize station
    def clean_station_name(s):
        if pd.isna(s):
            return s
        return str(s).strip()

    # Clean edge df station names
    route_edges_df["station_name_from"] = route_edges_df["station_name_from"].apply(clean_station_name)
    route_edges_df["station_name_to"] = route_edges_df["station_name_to"].apply(clean_station_name)
    station_feat_df["station_name"] = station_feat_df["station_name"].apply(clean_station_name)
    
    # Map alternate names to one standard name
    station_name_map = {
        "RamWadi": "Ramwadi",
        "Ruby Hall": "Ruby Hall Clinic",
        "Civil Court": "District Court (Civil Court)",
        "District Court Pune": "District Court (Civil Court)",
    }

    def apply_station_map(s):
        if pd.isna(s):
            return s
        return station_name_map.get(s, s)


    #encode string attributes
    #for every new value map to unique 
    line_map = {name: i for i, name in enumerate(route_edges_df["line"].unique())}     #line: purple | pink | aqua
    mode_map = {name: i for i, name in enumerate(route_edges_df["mode"].unique(), start=1)} # mode: metro | feeder bus

    route_edges_df["line_id"] = route_edges_df["line"].map(line_map)
    route_edges_df["mode_id"] = route_edges_df["mode"].map(mode_map)
    route_edges_df["is_transfer"] = route_edges_df["is_transfer"].astype(int)
    route_edges_df["bidirectional"] = route_edges_df["bidirectional"].astype(int)
    
    #check
    # print ("========route_edges table========")
    # print (route_edges_df)

    # print ("\n========timetable table========")
    # print (timetable_df)

    # print ("\n========swipes metadata table========")
    # print (swipes_df)
    
    #convert df to pyg df
    pyg_data = create_pyg_graph(route_edges_df,station_feat_df)
    
    #get start/final destation id from df
    # query = """
    # SELECT station_id_to
    # FROM route_edges_df
    # WHERE station_name_to = 'Nigdi'
    # """
    # station_id_to = sqldf(query, {"route_edges_df": route_edges_df})['station_id_to'][0]
    
    # query = """
    # SELECT station_id_from
    # FROM route_edges_df
    # WHERE station_name_from = 'Chinchwad'
    # """
    # station_id_from = sqldf(query, {"route_edges_df": route_edges_df})['station_id_from'][0]
    
    # station_id_to = route_edges_df.loc[
    # route_edges_df["station_name_to"] == "Nigdi", "station_id_to"
    # ].iloc[0]

    # station_id_from = route_edges_df.loc[
    #     route_edges_df["station_name_from"] == "Chinchwad", "station_id_from"
    # ].iloc[0]

    
    # print("-------station_id_from--------\n")
    # print(station_id_from)
    
    # print("-------station_id_to--------\n")
    # print(station_id_to)
    
        
    return route_edges_df, station_feat_df, pyg_data#, station_id_to,station_id_from

load_pyg_data_t("start_des", "final_des")



In [ ]:
from pipeline.pipelines import build_pyg_graph,run_multitask_inference, add_readable_labels, build_adjacency_list,generate_routes,score_routes
# from models.GAT.gat_journey_planner import create_pyg_graph

def recommend_routes_t(origin_station,destination_station,metro_edges_df,station_features_df,model, max_transfer):
    # Build graph
    pyg_data = build_pyg_graph(metro_edges_df, station_features_df)

    #Run GAT predictions
    predictions_df, _, _ = run_multitask_inference(model, pyg_data)
    
    # print("===========================\n")
    # print("predictions_df:\n")
    # print(predictions_df)
    predictions_df = add_readable_labels(predictions_df)

    #Build route adjacency
    adjacency = build_adjacency_list(metro_edges_df)

    #Generate possible routes
    possible_routes = generate_routes(adjacency,origin_station,destination_station,max_transfer,max_routes=5)

    if not possible_routes:
        return {
            "origin_station": origin_station,
            "destination_station": destination_station,
            "routes": [],
            "message": "No candidate routes found."
        }

    #Score routes
    rated_routes = score_routes(possible_routes, predictions_df)

    return {
        "origin_station": origin_station,
        "destination_station": destination_station,
        "routes": rated_routes
    }


## current testing 

In [ ]:
from pipeline.pipelines import recommend_routes, format_route_suggestions, normalize_trip_info, get_missing_fields,build_followup_question,congestion_penalty,feeder_bonus,get_Station_names
from models.GAT.gat_journey_planner import MultiTaskGAT
from datetime import datetime   
from models.llm.chatbot_agent import Chatbot, Extraction_SYS_MSG, FROMATTING_SYS_MSG

TRIP_SCHEMA = {
        "start_station_id": None,
        "end_station_id": None,
        "feeder_required": None,   # true / false / null
        "feeder_type": None,       # bike / bus / shuttle / null
        "departure_time": None,
        "arrival_time": None
        # "start_location": None,
        # "final_destination": None
    }
pro_dir = "C:\\Users\\sasab\\Documents\\Projects\\MaaS_AI\\Main_App\\"
REQUIRED_FIELDS = ["start_station", "end_station", "feeder_required"]
checkpoint_path = pro_dir+"models\\checkpoint\\gat_maas_model.pt"
data_path = "models\\GAT\\data\\"
   

def test():
    metro_edges_path = pro_dir+data_path+"pune_maas_journey_planner_data.csv"
    station_features_path = pro_dir+data_path+"station_features.csv"
    recieved_required = False
    extraction_bot = Chatbot(Extraction_SYS_MSG,model="qwen2.5:7b")
    formatter_bot = Chatbot(FROMATTING_SYS_MSG,model="qwen2.5:7b")
    normalize_res = ""
                
    #user input
    request_mess = """
“Let’s plan your trip\n\n
First, I’ll need a few details:

What station are you starting from?
What station are you heading to?
Do you need a feeder service (bike, bus, etc.)?\n

You can also include:
departure or arrival time
your exact starting or final destination
    """
    print(request_mess)
    
    user_input = "Need to get from Bhakti Shakti to Ruby Hall Clinic and yes i need a bike"#input("You: ")
    
    # loop until user exist convo
    #while True:  
        
    # user_input = input("You: ")
                    
    #loop until all required info is recieved
    while True: 
        
        if user_input.lower() in ["exit", "quit"]:
            break
                        
        bot_response = extraction_bot.chat(user_input)
        # print(f"Bot: {bot_response}\n")
        
        # print("RAW RESPONSE:", repr(response))
        
        normalize_res = normalize_trip_info(bot_response)
        missing_data, followup_required = get_missing_fields(normalize_res, REQUIRED_FIELDS)
                
        if not followup_required:
            print("All req info is obtained")
            break
        else:
            followup_questions = build_followup_question(missing_data)
            print("Bot:")
            print(followup_questions)
            user_input = input("You: ")
            
            # user_input = "follow up question:"+followup_questions + "and user response:"+ followup_rep
            # print(user_input)
    
    return normalize_res   
    # ------------


normalize_res = test()


In [ ]:
from pipeline.pipelines import load_pyg_data, load_trained_model

def recommend_routes_t(origin_station,destination_station,metro_edges_df,station_features_df,model):# max_transfer):
    # Build graph
    pyg_data = build_pyg_graph(metro_edges_df, station_features_df)

    #Run GAT predictions
    predictions_df, _, _ = run_multitask_inference(model, pyg_data)
    
    # print("===========================\n")
    # print("predictions_df:\n")
    # print(predictions_df)
    predictions_df = add_readable_labels(predictions_df)

    #Build route adjacency
    adjacency = build_adjacency_list(metro_edges_df)

    #Generate possible routes
    possible_routes = generate_routes(adjacency,origin_station,destination_station,max_routes=5)

    if not possible_routes:
        return {
            "origin_station": origin_station,
            "destination_station": destination_station,
            "routes": [],
            "message": "No candidate routes found."
        }

    #Score routes
    rated_routes = score_routes(possible_routes, predictions_df)

    return {
        "origin_station": origin_station,
        "destination_station": destination_station,
        "routes": rated_routes
    }
    

In [ ]:
from pipeline.pipelines import load_pyg_data,load_trained_model,build_pyg_graph,run_multitask_inference,add_readable_labels,build_adjacency_list,generate_routes
 # test input
start_station_name = normalize_res["start_station"] #"Bhakti Shakti"#input("Start station>")
final_station_name = normalize_res["end_station"] #"Shivaji Nagar"#input("End station>")
# print("-------station_from--------\n")
# print(start_station_name)

# print("-------station_to--------\n")
# print(final_station_name)

# # metro_edges_df, station_features_df, pyg_gat_data ,station_id_to,station_id_from= load_pyg_data()
metro_edges_df, station_features_df, pyg_gat_data = load_pyg_data()

station_id_from = metro_edges_df.loc[
    metro_edges_df["station_name_from"] == start_station_name, "station_id_from"
    ].iloc[0]

station_id_to = metro_edges_df.loc[
    metro_edges_df["station_name_to"] == final_station_name, "station_id_to"
    ].iloc[0]


print("-------TRIP SCHEMA--------\n")
TRIP_SCHEMA["start_station_id"] = station_id_from
TRIP_SCHEMA["end_station_id"] = station_id_to
TRIP_SCHEMA["feeder_required"] = normalize_res["feeder_required"]
TRIP_SCHEMA["feeder_type"] = normalize_res["feeder_type"]
TRIP_SCHEMA["departure_time"] = normalize_res["departure_time"]
TRIP_SCHEMA["arrival_time"] = normalize_res["arrival_time"]      
print(TRIP_SCHEMA)
        
model_kwargs ={
    "in_channels":pyg_gat_data.num_node_features,
    "hidden_channels":16, 
    "edge_features": pyg_gat_data.edge_attr.shape[1]
}

maas_gat_model, model_metadata = load_trained_model(MultiTaskGAT, checkpoint_path,model_kwargs)
    #for nb testing
    # print(result["routes"])
    # formatted_routes = format_route_suggestions(result)
    # print(formatted_routes)
    
    # # # bot = LocalChatbot(model="qwen2.5:7b")
    # # # #  while True:
    # # # #     user_input = input("You: ")
        
    # # # #     if user_input.lower() in ["exit", "quit"]:
    # # # #         break
            
    # # # #     response = bot.chat(user_input)
    # # # #     print(f"Bot: {response}\n")
    
    # user_input = "Take these route suggestions from the routing agent and write out a response for the user."+formatted_routes
    # print(user_input)
    # response = formatter_bot.chat(user_input)
    # print(f"Bot: {response}\n")
    
    # user_input = input("You:")

# result = test()


In [ ]:
# add for test
def score_routes_t(possible_routes, predictions_df):
    pred_map = predictions_df.set_index("station_id").to_dict("index")

    scored_routes = []

    for route in possible_routes:
        total_congestion_penalty = 0.0
        total_feeder_bonus = 0.0

        station_details = []

        for station_id in route:
            station_pred = pred_map.get(station_id, None)

            if station_pred is None:
                continue

            c_class = station_pred["pred_congestion_class"]
            f_class = station_pred["pred_feeder_class"]

            total_congestion_penalty += congestion_penalty(c_class)
            total_feeder_bonus += feeder_bonus(f_class)

            station_details.append({
                "station_id": station_id,
                "congestion_label": station_pred["congestion_label"],
                "feeder_label": station_pred["feeder_label"],
                "congestion_confidence": station_pred["pred_congestion_confidence"],
                "feeder_confidence": station_pred["pred_feeder_confidence"],
            })

        route_length_penalty = len(route) - 1

        final_score = route_length_penalty + total_congestion_penalty - total_feeder_bonus

        # print("-------------route---------------")
        # print(route)
        station_names_route = get_Station_names(route)
        # print("-------------station_names_route---------------")
        # print(station_names_route)
        
        scored_routes.append({
            "route": station_names_route,
            "num_stops": len(route),
            "route_length_penalty": route_length_penalty,
            "total_congestion_penalty": total_congestion_penalty,
            "total_feeder_bonus": total_feeder_bonus,
            "final_score": final_score,
            "station_details": station_details
        })

    scored_routes.sort(key=lambda r: r["final_score"])
    return scored_routes

pyg_data = build_pyg_graph(metro_edges_df, station_features_df)
# print("-------pyg_data--------\n")
# print(pyg_data)
        
#Run GAT predictions
pred_df, _, _ = run_multitask_inference(maas_gat_model, pyg_data)
pred_df = add_readable_labels(pred_df)
# print("-------pred_df--------\n")
# print(pred_df)

#Build route adjacency
adjacency = build_adjacency_list(metro_edges_df)
# print("-------adjacency--------\n")
# print(adjacency)

#Generate possible routes
possible_routes_ls = generate_routes(adjacency,TRIP_SCHEMA["start_station_id"],TRIP_SCHEMA["end_station_id"],max_routes=5)
# print("-------possible_routes_ls--------\n")
# print(possible_routes_ls)
# print("end")

ranked_routes = score_routes_t(possible_routes_ls, pred_df)

print(ranked_routes)

P01
P02
P03
P04
P05
P06
P07
P08
P09
P10
P11
P12
P13
T002
T001
A12
A13
A14
P01
P02
P03
P04
P05
P06
P07
P08
P09
P10
P11
P12
P13
T002
T001
A12
A13
A14
[{'route': [['Bhakti Shakti', 'Nigdi', 'Akurdi', 'Chinchwad', 'PCMC Bhavan', 'Sant Tukaram Nagar', 'Nashik Phata', 'Kasarwadi', 'Phugewadi', 'Dapodi', 'Bopodi', 'Khadki', 'Range Hills', 'Shivaji Nagar', 'District Court (Civil Court)', 'Mangalwar Peth', 'Pune Railway Station', 'Ruby Hall Clinic']], 'num_stops': 18, 'route_length_penalty': 17, 'total_congestion_penalty': 0.0, 'total_feeder_bonus': 0.0, 'final_score': 17.0, 'station_details': [{'station_id': 'P01', 'congestion_label': 0.0, 'feeder_label': 0.0, 'congestion_confidence': 0.9981739521026611, 'feeder_confidence': 0.998769223690033}, {'station_id': 'P02', 'congestion_label': 0.0, 'feeder_label': 0.0, 'congestion_confidence': 0.9981051683425903, 'feeder_confidence': 0.9987533092498779}, {'station_id': 'P03', 'congestion_label': 0.0, 'feeder_label': 0.0, 'congestion_confidence': 0.997

In [ ]:
from pipeline.pipelines import load_station_master
def get_Station_names_t(route):
    stations = []
    
    station_master_df = load_station_master()

    # for route_info in ranked_sugg_routes:
    #     route_path = route_info #route_info["route"]
    station_name = []
        
        # for station_id in route_path:
    for index in range(len(route)):      
        station_id = route[index]
        # print(station_id)
        
        try: 
            station_name.append(station_master_df.loc[
                station_master_df["master_station_id"] == station_id, "station_name"
                ].iloc[0])
            
        except IndexError:
            # No match found
            # return "Station id: {station_id} was not found"
            print(f"Station id: {station_id} was not found")
        # if index == len(route)-1:#last index
        #     # print(station_name)
        #     stations.append(station_name)
            
    # print(station_name)
    return station_name

test = ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'T002', 'T001', 'A12', 'A13', 'A14']
st = get_Station_names_t(test)


['Bhakti Shakti', 'Nigdi', 'Akurdi', 'Chinchwad', 'PCMC Bhavan', 'Sant Tukaram Nagar', 'Nashik Phata', 'Kasarwadi', 'Phugewadi', 'Dapodi', 'Bopodi', 'Khadki', 'Range Hills', 'Shivaji Nagar', 'District Court (Civil Court)', 'Mangalwar Peth', 'Pune Railway Station', 'Ruby Hall Clinic']


In [24]:
# end add
result = recommend_routes(
origin_station=TRIP_SCHEMA["start_station_id"],
destination_station=TRIP_SCHEMA["end_station_id"],
metro_edges_df=metro_edges_df,
station_features_df=station_features_df,
model=maas_gat_model
)

formatted_routes = format_route_suggestions(result)
print(formatted_routes)

Station id: P was not found
Station id: 0 was not found
Station id: 1 was not found
Station id: P was not found
Station id: 0 was not found
Station id: 2 was not found
Station id: P was not found
Station id: 0 was not found
Station id: 3 was not found
Station id: P was not found
Station id: 0 was not found
Station id: 4 was not found
Station id: P was not found
Station id: 0 was not found
Station id: 5 was not found
Station id: P was not found
Station id: 0 was not found
Station id: 6 was not found
Station id: P was not found
Station id: 0 was not found
Station id: 7 was not found
Station id: P was not found
Station id: 0 was not found
Station id: 8 was not found
Station id: P was not found
Station id: 0 was not found
Station id: 9 was not found
Station id: P was not found
Station id: 1 was not found
Station id: 0 was not found
Station id: P was not found
Station id: 1 was not found
Station id: 1 was not found
Station id: P was not found
Station id: 1 was not found
Station id: 2 was no

In [ ]:
import pandas as pd
from pipeline.pipelines import load_station_master

sugg_routes_info_ls = result["routes"]
# pro_dir = "C:\\Users\\sasab\\Documents\\Projects\\MaaS_AI\\Main_App\\"
data_path = "models\\GAT\\data\\"
# print(sugg_routes_info_ls)
# fix differnt name - normalize station
# Map alternate names to one standard name
station_name_map = {
    "RamWadi": "Ramwadi",
    "Ruby Hall": "Ruby Hall Clinic",
    "Civil Court": "District Court (Civil Court)",
    "District Court Pune": "District Court (Civil Court)",
}

def get_Station_names(ranked_sugg_routes):
    stations = []
    
    station_master_df = load_station_master()

    for route_info in ranked_sugg_routes:
        route_path = route_info["route"]
        station_name = []
        
        # for station_id in route_path:
        for index in range(len(route_path)):      
            station_id = route_path[index]
            # print(station_id)
            
            try: 
                station_name.append(station_master_df.loc[
                    station_master_df["master_station_id"] == station_id, "station_name"
                    ].iloc[0])
                
            except IndexError:
                # No match found
                # return "Station id: {station_id} was not found"
                print(f"Station id: {station_id} was not found")
            
            if index == len(route_path)-1:#last index
                # print(station_name)
                stations.append(station_name)
                
    return stations    


st = get_Station_names(sugg_routes_info_ls) 
print(st)


# formatted_routes = format_route_suggestions(result)
# print(formatted_routes)